# 1. CSV 파일 가공

- 과거 경기 결과 데이터와 과거 피파 랭킹 데이터를 합쳐서, 모델이 학습에 적합하도록 가공된 데이터프레임을 만듭니다.
- 너무 오래전 기록은 의미가 없기에 2018년 러시아 월드컵 직후부터 2026북중미 월드컵 직전까지의 데이터를 사용합니다.
  
##### pd.merge_asof
경기 날짜와 피파 랭킹 발표일은 정확히 일치하지 않습니다. 경기 당일 기준 가장 최근 데이터 랭킹을 매핑하기 위해 이중 for문을 쓰면 연산이 비효율적입니다.
이를 해결하기 위해 Pnadas의 merge_asof()를 사용했습니다. 국가명(by)를 기준으로 데이터를 분할한 후 시계열 정렬된 상태에서 탐색을 수행하여 병합 속도를 최적화했습니다.


In [10]:
import pandas as pd
import numpy as np

# 1. 파일 불러오기
results = pd.read_csv('data/results.csv')
rankings = pd.read_csv('data/fifa_ranking-2024-04-04.csv')


# 2. 날짜 대소 비교 연산을 위해 datetime(시계열) 타입으로 변환
results['date'] = pd.to_datetime(results['date'])
rankings['rank_date'] = pd.to_datetime(rankings['rank_date'])

# 3. 데이터 필터링 (2018 월드컵 직후 ~ 2026 북중미 월드컵 직전)
start_date = '2018-08-01'
worldcup_start_date = '2026-06-11'

# .copy()를 써서 원본 데이터의 view가 아닌 독립적인 메모리를 할당함. (독립적인 DataFrame 객체)
train_data = results[(results['date'] >= start_date) & (results['date'] < worldcup_start_date)].copy()

# 4. merge_asof를 위한 시간순(오름차순) 정렬
train_data = train_data.sort_values('date')
rankings = rankings.sort_values('rank_date')

# 5. 홈팀 랭킹 병합 (2024~2026 경기는 2024년 마지막 랭킹으로 적용)
# merge_asof: 'date'열을 기준으로 가장 가까운 과거(backward)의 피파 랭킹 매핑
# by 파라미터로 국가를 그룹화한 뒤 내부적으로 two pointer 탐색을 수행하여 병합함 (이를 위해 시간순 정렬)
train_data = pd.merge_asof(
    train_data,            # 왼쪽 기준표 (경기 데이터)
    rankings,              # 오른쪽 병합표 (피파 랭킹)
    left_on='date',        # 기준 시간
    right_on='rank_date',  # 병합할 시간
    left_by='home_team',   # 그룹화 조건 (홈팀), 두 by 값이 같은 것끼리만 매칭됨
    right_by='country_full',
    direction='backward'   # 시간 탐색 방향 (경기일 이전의 가장 최근 랭킹 매핑 위해)
)
train_data = train_data.rename(columns={'rank': 'home_rank'}) # 'home_team' 기준으로 랭크를 이어 붙였으니 rank를 'home_rank'로 이름을 바꿔줌
train_data = train_data[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'home_rank']]

# 6. 어웨이팀 랭킹 합치기
train_data = pd.merge_asof(
    train_data,
    rankings,
    left_on='date',
    right_on='rank_date',
    left_by='away_team',
    right_by='country_full',
    direction='backward'
)
train_data = train_data.rename(columns={'rank': 'away_rank'})
train_data = train_data[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'home_rank', 'away_rank']]

# 7. 결측치 제거
# 과거 랭킹 데이터가 없는 신생 국가 등 노이즈 제거
train_data = train_data.dropna()

print(f"최종 학습 데이터 수: {len(train_data)}개")
display(train_data.head())
display(train_data.tail())

최종 학습 데이터 수: 6197개


,date,home_team,away_team,home_score,away_score,home_rank,away_rank
0,2018-08-04,Belize,Barbados,1.0,0.0,163.0,162.0
1,2018-08-04,Palestine,Iraq,0.0,3.0,101.0,84.0
2,2018-08-15,Guatemala,Cuba,3.0,0.0,133.0,168.0
3,2018-08-18,Andorra,United Arab Emirates,0.0,0.0,130.0,77.0
4,2018-08-18,Grenada,Jamaica,1.0,5.0,168.0,54.0


,date,home_team,away_team,home_score,away_score,home_rank,away_rank
7693,2026-06-09,Hungary,Kazakhstan,3.0,1.0,26.0,103.0
7694,2026-06-10,Portugal,Nigeria,2.0,1.0,6.0,30.0
7695,2026-06-10,Bolivia,Algeria,0.0,4.0,85.0,43.0
7696,2026-06-10,England,Costa Rica,3.0,0.0,4.0,52.0
7697,2026-06-10,Afghanistan,Pakistan,0.0,2.0,151.0,195.0


# 2. Feautre Engineering
- ML 모델에 데이터를 넣을 수 있도록 위의 데이터프레임 데이터를 가공합니다.
- 정답지에 해당하는 Target
- Feature 1에 해당하는 rank_diff (랭크 차이)
- Feature 2에 해당하는 gd_diff (최근 3경기 득실차)
- 결과적으로 두 팀간의 rank_diff, gd_diff에 따른 승패 여부를 학습하게 됩니다.

In [11]:
import pandas as pd
import numpy as np

# 1. Target 열 생성 (홈팀 승리=1, 무승부/패배=0)
train_data['target'] = (train_data['home_score'] > train_data['away_score']).astype(int)

# 2. Feature 1: rank_diff 열 생성
# 각 팀의 랭크 수치를 차의 형태로 만들어 모델 학습에 용이하게 함
train_data['rank_diff'] = train_data['home_rank'] - train_data['away_rank']

# 3. Feature 2: gd_diff 열 생성 (최근 3경기 득실차)
# 경기 중심 데이터를 팀 중심 시계열 데이터로 변환.

# home_team 기준으로 team, gf: 득점, ga: 실점 표 생성
home_stats = train_data[['date', 'home_team', 'home_score', 'away_score']].rename(
    columns={'home_team': 'team', 'home_score': 'gf', 'away_score': 'ga'})

# away_team 기준으로 team, gf, ga 표 (away team 입장에선 away_score가 gf가 됨)
away_stats = train_data[['date', 'away_team', 'away_score', 'home_score']].rename(
    columns={'away_team': 'team', 'away_score': 'gf', 'home_score': 'ga'})

# home_team, away_team 구분 없이 team, gf, ga 기록을 세로로 병합함. 그리고 팀별 gd(득실차) 계산
all_stats = pd.concat([home_stats, away_stats]).sort_values(['team', 'date'])
all_stats['gd'] = all_stats['gf'] - all_stats['ga']

# Data Leakage 방지를 고려하여 모든 팀별로 매치일 기준 최근 3경기 골득실 기록
# - shift(1): 오늘 경기 결과가 과거 폼 계산에 들어가지 않도록 한 칸씩 뒤로 미룸
# - rolling(3, min_periods=1): 최근 3경기 합산 (3경기가 안 채워졌어도 있는 만큼 합산)
# - fillna(0.0): 첫 번째 경기라 과거 기록이 아예 없는 경우 결측치 대신 기본값 0.0 부여
all_stats['recent_3_gd'] = all_stats.groupby('team')['gd'].transform(
    lambda x: x.rolling(window=3, min_periods=1).sum().shift(1).fillna(0.0)
)

# 4. 계산된 최근 3경기 골득실 데이터를 원본 train_data에 병합
# all_stats의 date와 train_data의 date는 일치하기에 pd.merge사용해서 병합
# home_team 폼
train_data = pd.merge(train_data, all_stats[['date', 'team', 'recent_3_gd']], 
                      left_on=['date', 'home_team'], right_on=['date', 'team'], how='left').drop(columns=['team'])
train_data = train_data.rename(columns={'recent_3_gd': 'home_recent_gd'})
# date, home_team 두 조건을 매칭하여 병합을 진행함. 데이터 손실을 막기 위해 all_stats에 데이터가 없으면 NaN으로 처리하도록 함 (ex: 이전에 치른 3경기가 없을 때)

# away_team 폼
train_data = pd.merge(train_data, all_stats[['date', 'team', 'recent_3_gd']], 
                      left_on=['date', 'away_team'], right_on=['date', 'team'], how='left').drop(columns=['team'])
train_data = train_data.rename(columns={'recent_3_gd': 'away_recent_gd'})

# 5. 두 팀 간의 최근 폼(최근 3경기 득실차) 차이 계산
train_data['gd_diff'] = train_data['home_recent_gd'] - train_data['away_recent_gd']

# 병합 과정에서 발생할 수 있는 결측치 제거
train_data = train_data.dropna()

# 모델 입력용 X, y 추출. Numpy 배열로 추출.
# X: (n_samples, n_features) / y: (n_samples, )
X = train_data[['rank_diff', 'gd_diff']].values # 2 Features
y = train_data['target'].values

print(f"X shape:", X.shape)
display(train_data[['home_team', 'away_team', 'rank_diff', 'gd_diff', 'target']].head())
display(train_data[['home_team', 'away_team', 'rank_diff', 'gd_diff', 'target']].tail())

X shape: (6207, 2)


,home_team,away_team,rank_diff,gd_diff,target
0,Belize,Barbados,1.0,0.0,1
1,Palestine,Iraq,17.0,0.0,0
2,Guatemala,Cuba,-35.0,0.0,1
3,Andorra,United Arab Emirates,53.0,0.0,0
4,Grenada,Jamaica,114.0,0.0,0


,home_team,away_team,rank_diff,gd_diff,target
6202,Hungary,Kazakhstan,-77.0,-1.0,1
6203,Portugal,Nigeria,-24.0,4.0,1
6204,Bolivia,Algeria,42.0,-12.0,0
6205,England,Costa Rica,-48.0,2.0,1
6206,Afghanistan,Pakistan,-44.0,-1.0,0


# 3. 모델 학습
- 위에서 Feature Engineering 완료한 데이터셋을 ML 모델에 넣어 logistic, MLP 모델을 학습시킴
- target 클래스가 0, 1 뿐이기에 두 모델 모두 binary 분류를 수행하게 됨

In [12]:
from ai_lab.model_selection import train_test_split
from ai_lab.preprocessing import StandardScaler
from ai_lab.linear_model import LogisticRegression
from ai_lab.neural_network import MLPClassifier

# 1. 데이터 분할 및 스케일링
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Logistic Regression
log_model = LogisticRegression(lr=0.01, epochs=3000, random_state=42)
log_model.fit(X_train_scaled, y_train)

# 3. MLP Classifier
# - batch_size=32 추가 (미니배치로 잘게 쪼개서 학습시켜야 가중치가 역동적으로 변함)
# - 미니배치를 쓰면 가중치 업데이트가 잦아지므로 epochs는 200 정도로 줄임
# - activation='tanh' (데이터 scale이 작을 때 relu보다 뉴런이 죽는 현상이 덜 발생함)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(32, 16), 
    activation='tanh',  # ReLU 대신 tanh 사용
    lr=0.01,            # 학습률 안정화
    epochs=200,         # 미니배치이므로 에포크 축소
    batch_size=32,      # 핵심! 미니배치 적용
    random_state=42
)
mlp_model.fit(X_train_scaled, y_train)

# 4. 평가 (ClassifierMixin에서 상속받은 score() 메서드 사용)
log_test_acc = log_model.score(X_test_scaled, y_test)
mlp_test_acc = mlp_model.score(X_test_scaled, y_test)

print("=== 모델 성능 비교 (Test Accuracy) ===")
print(f"[Logistic Regression] : {log_test_acc:.4f}")
print(f"[MLP Classifier]      : {mlp_test_acc:.4f}")

# 5. 확률 비교
log_proba = log_model.predict_proba(X_test_scaled[:5])
mlp_proba = mlp_model.predict_proba(X_test_scaled[:5])

print("\n=== 예측 확률 비교 (Test Data 첫 5경기 홈팀 승률) ===")
for i in range(5):
    
    print(f"경기 {i+1} | Logistic: {log_proba[i]:.4f} | MLP: {mlp_proba[i][0]:.4f}")

=== 모델 성능 비교 (Test Accuracy) ===
[Logistic Regression] : 0.7204
[MLP Classifier]      : 0.7196

=== 예측 확률 비교 (Test Data 첫 5경기 홈팀 승률) ===
경기 1 | Logistic: 0.0638 | MLP: 0.0675
경기 2 | Logistic: 0.8093 | MLP: 0.8157
경기 3 | Logistic: 0.0664 | MLP: 0.0711
경기 4 | Logistic: 0.4476 | MLP: 0.4509
경기 5 | Logistic: 0.6374 | MLP: 0.6462


# 4. 모델 추론 및 Monte Carlo Simulation 환경 구축
- 학습이 끝난 모델을 조별리그 및 32강 데이터에 적용하여 예측하는 함수 구축
- 이후 이어질 Monte Carlo Simulation에 사용할 확률 유닛 생성. 

In [15]:
import numpy as np
import pandas as pd

# 1. 16강 진출 팀 데이터 로드
ro16_df = pd.read_csv('data/ro16_teams.csv')
ro16_df = ro16_df.set_index('Team_Name') # 팀명을 인덱스로 세팅하여 검색하기 편하게 만듦.

# 2. 두 팀이 맞붙었을 때 승자를 예측하는 함수
def predict_match_winner(team_A, team_B, model, scaler):
    """
    team_A와 team_B의 데이터를 모델에 넣고, 승률을 기반으로 주사위를 굴려 승자를 반환함.

    Args:
        team_A (str): 홈팀 (Team A)
        team_B (str): 어웨이팀 (Team B)
        model: 예측에 사용할 학습된 모델 (LogisticRegression or MLPClassifier)
        scaler: 데이터 전처리에 사용했던 StandardScaler 객체
        
    Returns:
        str: 시뮬레이션 결과 승리한 팀의 이름

    """
    # 1) 두 팀의 데이터(피파랭킹, 최근 폼)를 가져옴
    # Group_GD가 아까 모델 학습할 때 썼던 'recent_3_gd'와 동일한 역할(최근 폼)을 함
    # 팀 이름을 인덱스로 세팅했기에 .loc(행, 열)로 데이터 접근 가능
    rank_A = ro16_df.loc[team_A, 'FIFA_Rank']
    rank_B = ro16_df.loc[team_B, 'FIFA_Rank']
    form_A = ro16_df.loc[team_A, 'Group_GD']
    form_B = ro16_df.loc[team_B, 'Group_GD']
    
    # 2) 모델에 넣을 Input(Feature) 모양으로 계산해줌.
    # 학습할 때 [rank_diff, gd_diff] 순서로 넣었으므로 순서를 똑같이 맞춰야 함.
    rank_diff = rank_A - rank_B
    gd_diff = form_A - form_B
    
    # 3) 스케일링 (학습할 때 썼던 scaler를 그대로 사용해야 함)
    # X_input은 (n_samples, 2) 형태의 2차원 배열이어야 하므로 [[ ]] 로 감싸줌
    X_input = np.array([[rank_diff, gd_diff]])
    X_input_scaled = scaler.transform(X_input)
    
    # 4) 승률 예측
    # Logistic(1D)과 MLP(2D) 모델의 출력 차원이 다르기에, flatten()을 통해 차원 오류 방지하고 스칼라값만 추출
    proba = model.predict_proba(X_input_scaled)
    prob_A_wins = float(proba.flatten()[0]) # binary class이기에 proba의 반환값은 A팀이 승리(Class 1)할 확률
    
    # 5) 가중치 보정
    # ML 모델 학습 시 사용하지 않은 데이터(최근 몸값)를 수식으로 반영해 승률을 보정함.
    value_A = ro16_df.loc[team_A, 'Total_squad_Value']
    value_B = ro16_df.loc[team_B, 'Total_squad_Value']
    
    # 몸값 차이가 클수록 승률에 어드밴티지를 줌 (수식은 직관적으로 설정했음)
    # 예: A몸값이 B보다 2배 높으면 승률 5% 증가 (가중치 폭은 임의 조정 가능)
    value_ratio = value_A / (value_B + 1e-5) # 0으로 나누기 방지
    
    if value_ratio > 1.5: # A팀이 1.5배 이상 비싸다면 0.05 (5%)의 어드밴티지 더해줌
        prob_A_wins = min(prob_A_wins + 0.05, 0.99) # 최대 확률 99% 제한
    elif value_ratio < 0.66: # A팀이 1.5배 이상 싸다면
        prob_A_wins = max(prob_A_wins - 0.05, 0.01) # 최소 확률 1% 제한

    # 32강 토너먼트 결과에 따른 가중치 보정
    # 32강에서의 득실차를 구해 어드밴티지 부여
    r_32_gd_A = ro16_df.loc[team_A, 'R32_GF'] - ro16_df.loc[team_A, 'R32_GA']
    r_32_gd_B = ro16_df.loc[team_B, 'R32_GF'] - ro16_df.loc[team_B, 'R32_GA']

    # 두 팀의 32강 득실차 비교
    gd_32_diff = r_32_gd_A - r_32_gd_B

    # 득실차 1골당 2%(0.02)씩 어드밴티지 부여 (수식은 직관적으로 설정했음)

    prob_A_wins += (gd_32_diff * 0.02)
    prob_A_wins = np.clip(prob_A_wins, 0.01, 0.99) # 확률 범위 벗어나지 않도록 조절
        
    # 6) 난수(주사위) 발생으로 승자 결정 (이변 반영)
    random_dice = np.random.rand() # 0.0 ~ 1.0 사이 난수생성
    
    if random_dice < prob_A_wins:
        return team_A  # A팀 승리
    else:
        return team_B  # B팀 승리 (무승부는 승부차기 끝에 B승리라고 가정)

# 테스트 실행 
winner = predict_match_winner('Argentina', 'Egypt', mlp_model, scaler)
print(f"테스트 매치 승자: {winner}")

테스트 매치 승자: Argentina


# 5. Monte Carlo Simulation 진행 및 각 팀별 최종 우승확률 도출

In [17]:
import numpy as np
import pandas as pd

# 1. 실제 16강 대진표 세팅 (리스트 안의 튜플 형태)
matchups_16 = [
    ('Canada', 'Morocco'),    # 매치 1
    ('Paraguay', 'France'),        # 매치 2 (이후 매치1 승자와 매치2 승자가 8강에서 만남)
    ('United States', 'Belgium'),      # 매치 3
    ('Portugal', 'Spain'),         # 매치 4 (매치3 승자 vs 매치4 승자)
    ('Brazil', 'Norway'),  # 매치 5
    ('Mexico', 'England'),        # 매치 6 (매치5 승자 vs 매치6 승자)
    ('Switzerland', 'Colombia'),# 매치 7
    ('Argentina', 'Egypt')           # 매치 8 (매치7 승자 vs 매치8 승자)
]

def run_tournament(bracket, model, scaler):
    """
    16강 대진표를 입력받아 결승전까지 모든 경기를 시뮬레이션하고 최종 우승팀을 반환함.

    Args:
        bracket (list[tuple[str, str]]): 맞붙는 두 국가의 이름이 담긴 튜플들의 리스트 (16강 대진표)
        model (ClassifierMixin): 승률 예측에 사용할 학습이 완료된 모델 (MLP or Logistic)
        scaler (StandardScaler): 데이터 스케일링에 사용할 학습이 완료된 전처리기 객체

    Returns:
        str: 시뮬레이션 결과 최종 우승을 차지한 국가의 이름
    """
    # 16강 진행 -> 8팀 생존
    qf_teams = [] # 8강 진출팀을 담을 리스트. 위에서 구현한 시뮬레이션 기능 이용
    for team_A, team_B in bracket:
        winner = predict_match_winner(team_A, team_B, model, scaler)
        qf_teams.append(winner)
        
    # 8강 진행 (인접한 2 팀끼리 대결) -> 4팀 생존
    sf_teams = [] # 4강 진출팀을 담을 리스트
    for i in range(0, 8, 2):
        winner = predict_match_winner(qf_teams[i], qf_teams[i+1], model, scaler)
        sf_teams.append(winner)
        
    # 4강(준결승) 진행 -> 2팀 생존
    final_teams = [] # 결승 진출팀을 담을 리스트
    for i in range(0, 4, 2):
        winner = predict_match_winner(sf_teams[i], sf_teams[i+1], model, scaler)
        final_teams.append(winner)
        
    # 결승전 진행 -> 1팀 우승
    champion = predict_match_winner(final_teams[0], final_teams[1], model, scaler)
    return champion

# 2. 몬테카를로 시뮬레이션 10,000회 실행. 확률 유닛의 편향 방지 위해서 여러 번 반복을 통해 신뢰도 높은 예측 결과를 얻기 위함 (적은 수의 반복으로 이변이 여러 번 발생해 잘못된 확률 방지)
n_simulations = 10000

# 16개 나라의 우승 힛수를 기록할 딕셔너리 생성 (모두 0으로 초기화)
championship_counts = {team: 0 for team in ro16_df.index}

print(f"[{n_simulations}회] 월드컵 토너먼트 시뮬레이션 시작...")

# 선택한 모델로 1만 번의 월드컵을 진행함
for i in range(n_simulations):
    champ = run_tournament(matchups_16, mlp_model, scaler)
    championship_counts[champ] += 1
    
    # 진행 상황 출력
    if (i + 1) % 2000 == 0:
        print(f"... {i + 1}회 완료")

print("시뮬레이션 종료\n")

# 3. 결과 집계 및 확률 계산
# 딕셔너리를 DataFrame 객체로 변환. 국가 이름(Key)을 인덱스로, 우승 횟수(Value)를 'Wins'열로 추가
results_df = pd.DataFrame.from_dict(championship_counts, orient='index', columns=['Wins'])

# 확률 계산하기: (우승 횟수 / 전체 10000번) * 100. 벡터 계산을 통해 Win_Probablity라는 새로운 열의 데이터로 추가
results_df['Win_Probability (%)'] = (results_df['Wins'] / n_simulations) * 100

# 우승 확률이 높은 국가가 맨 위에 오도록 정렬
results_df = results_df.sort_values(by='Win_Probability (%)', ascending=False)

# 최종 우승 확률표 출력
print("=== 2026 월드컵 최종 우승 확률 ===")
display(results_df[results_df['Win_Probability (%)'] > 0]) # 우승 확률이 0%가 아닌 팀만 출력

[10000회] 월드컵 토너먼트 시뮬레이션 시작...
... 2000회 완료
... 4000회 완료
... 6000회 완료
... 8000회 완료
... 10000회 완료
시뮬레이션 종료

=== 2026 월드컵 최종 우승 확률 ===


,Wins,Win_Probability (%)
France,2247,22.47
Argentina,1750,17.50
Spain,1421,14.21
England,1081,10.81
Brazil,799,7.99
Portugal,555,5.55
Colombia,448,4.48
Mexico,385,3.85
Belgium,334,3.34
Switzerland,285,2.85
